# LLM Hallucination Detection Tutorial

## What are LLM Hallucinations?

**LLM Hallucinations** occur when a Large Language Model (like ChatGPT, Claude, or GPT-4) generates information that is:
- **False or misleading** - The model claims something that isn't true
- **Not supported by training data** - The model "makes up" information
- **Inconsistent with facts** - The model contradicts known information

### Why Do Hallucinations Matter?

Hallucinations are a major concern because:
- They can spread misinformation
- They reduce trust in AI systems
- They can lead to poor decision-making
- They're particularly dangerous in domains like healthcare, legal, or scientific research

### Example of a Hallucination

If you ask an LLM about a book that doesn't exist, it might:
- Invent a fake author
- Create a fake plot summary
- Provide fake publication dates
- Even give fake reviews

This notebook will show you how to detect and analyze hallucinations in LLM responses using the **Blueprint** framework and **Giskard** scanning tools.

## Setup and Imports

First, we need to import the necessary libraries and tools:

- **loguru**: For better logging and debugging
- **giskard**: A framework for testing and scanning AI models
- **pandas**: For data manipulation
- **blueprint**: Our custom framework for building and testing chatbots

In [ ]:
from loguru import logger
import giskard
import pandas as pd
from blueprint.chatbot import Chatbot
from blueprint.settings import OUTPUT_FOLDER, IPCC_REPORT_URL, PROMPT_TEMPLATE

## Creating Our Test Model

In this tutorial, we'll create a **Climate Change Question Answering** model that:

1. **Uses IPCC reports as knowledge base** - IPCC (Intergovernmental Panel on Climate Change) reports are authoritative scientific documents
2. **Answers questions about climate change** - This is a domain where accuracy is crucial
3. **Will be tested for hallucinations** - We'll see if it makes up information not in the IPCC reports

### What Each Component Does:

- **Chatbot**: Our custom chatbot that reads PDF documents and answers questions
- **giskard.Model**: Wraps our chatbot so it can be scanned for vulnerabilities
- **model_type="text_generation"**: Tells Giskard this is a text generation model
- **feature_names**: Specifies that our input is a "question"

In [ ]:
bot = Chatbot(
        pdf=IPCC_REPORT_URL,
        prompt_template=PROMPT_TEMPLATE,
        local=False,
        output_folder=OUTPUT_FOLDER,
    )
giskard_model = giskard.Model(
        model=bot.predict,
        model_type="text_generation",
        name="Climate Change Question Answering",
        description="This model answers any question about climate change based on IPCC reports",
        feature_names=["question"],
    )

## Creating Test Questions

Now we'll create a dataset of questions to test our model. These questions are designed to:

- **Test factual accuracy** - Questions that should have specific answers in IPCC reports
- **Identify potential hallucinations** - If the model answers with information not in the reports
- **Cover different aspects** - Various topics within climate change

### Why This Question?

The question "According to the IPCC report, what are key risks in Europe?" is good for testing because:
- It's specific to a geographic region
- It asks for "key risks" which should be well-documented
- It's factual rather than speculative
- We can verify the answer against the actual IPCC report

In [ ]:
examples = [
        "According to the IPCC report, what are key risks in the Europe?",
        # "Is sea level rise avoidable? When will it stop?",
]
giskard_dataset = giskard.Dataset(pd.DataFrame({"question": examples}), target=None)

## Testing Our Model

Let's first see how our model responds to our test question. This will help us understand:

- **What the model outputs** - The actual answer it gives
- **The quality of the response** - Whether it seems accurate and detailed
- **Potential issues** - Any red flags that might indicate hallucinations

### Analyzing the Response

When you see the model's answer, ask yourself:
- Does this sound like information from a scientific report?
- Are the details specific and factual?
- Does it match what you'd expect from IPCC reports?
- Are there any claims that seem suspicious or too specific?

In [ ]:
answers = giskard_model.predict(giskard_dataset).prediction
logger.info([f"\n{q}: {a}" for q, a in zip(examples, answers)])

## Running the Hallucination Scan

Now we'll use Giskard's scanning capabilities to systematically test our model for hallucinations. This is the core of our vulnerability detection.

### What the Scan Does:

The `giskard.scan()` function will:

1. **Generate test cases** - Create variations of our questions to test different scenarios
2. **Run multiple detectors** - Use different algorithms to identify potential hallucinations
3. **Analyze responses** - Compare model outputs against expected behavior
4. **Generate a report** - Provide detailed findings about potential issues

### Types of Detectors:

- **LLMBasicSycophancyDetector**: Tests if the model just agrees with whatever the user says
- **LLMImplausibleOutputDetector**: Identifies responses that seem unrealistic or impossible

### Estimated Calls:

The scan will make approximately 30 calls to your model and 22 calls to evaluation LLMs. This is necessary for thorough testing.

In [ ]:
full_report = giskard.scan(giskard_model, giskard_dataset, only="hallucination")

## Viewing the Scan Results

The scan results will show you:

- **Issues found** - Any potential hallucinations or vulnerabilities detected
- **Test cases** - The specific questions that revealed problems
- **Severity levels** - How serious each issue is
- **Recommendations** - How to fix or improve the model

### Understanding the Results:

Look for:
- **High severity issues** - These are the most concerning
- **Patterns** - Do certain types of questions consistently cause problems?
- **False positives** - Sometimes the scanner might flag legitimate responses
- **Actionable insights** - What can you actually do to improve the model?

In [ ]:
display(full_report)

## Exporting Results

We can export the scan results in different formats for further analysis or sharing:

### HTML Report
Creates an interactive web page with the full scan results, including visualizations and detailed explanations.

In [ ]:
html = full_report.to_html("report.html", embed=True)

### JSON Export
Saves the results in JSON format, which is useful for:
- Programmatic analysis
- Integration with other tools
- Automated processing

In [ ]:
json_path = OUTPUT_FOLDER / "scan_report.json"
json_report = full_report.to_json(filename=json_path)
logger.info(f"Exported to {json_path}")

### Markdown Export
Creates a markdown file that's perfect for:
- Documentation
- Sharing with non-technical stakeholders
- Version control
- Easy reading on platforms like GitHub

In [ ]:
md_path = OUTPUT_FOLDER / "scan_report.md"
md_report = full_report.to_markdown(filename=md_path, template="huggingface")
logger.info(f"Exported to {md_path}")

## Viewing the Markdown Report

Let's see what the markdown report looks like. This format is particularly useful for documentation and sharing results.

In [ ]:
md_report

## Additional Export Options

You can also export the report directly to the current directory for easy access.

In [ ]:
full_report.to_json("report.json")

## Summary and Next Steps

### What We've Learned:

1. **What hallucinations are** - False or misleading information generated by LLMs
2. **Why they matter** - They can spread misinformation and reduce trust
3. **How to detect them** - Using systematic scanning tools like Giskard
4. **How to analyze results** - Understanding scan reports and their implications

### Key Takeaways:

- **Always test your LLM applications** - Don't assume they're accurate
- **Use systematic approaches** - Manual testing isn't enough for complex systems
- **Document your findings** - Keep records of vulnerabilities and fixes
- **Iterate and improve** - Use scan results to make your models better

### Further Exploration:

You can extend this tutorial by:
- Testing different types of questions
- Comparing multiple models
- Adding more sophisticated detection methods
- Implementing fixes for identified issues

### Real-World Applications:

This type of testing is crucial for:
- **Healthcare AI** - Where accuracy can be life-or-death
- **Legal AI** - Where misinformation can have serious consequences
- **Educational AI** - Where students need accurate information
- **Research AI** - Where scientific accuracy is paramount

Remember: **The goal isn't to eliminate all hallucinations** (which may be impossible), but to **understand and manage them** so your AI applications are as reliable as possible.